# E-Commerce Data Warehouse Pipeline

This notebook implements a complete data warehouse pipeline for the Udacity Data Engineering course.

**Business Context:** You are a data engineer at a fast-growing e-commerce company. Critical data is spread across multiple operational systems (PostgreSQL, Cassandra, Neo4j), making it difficult for analysts to run consistent reports. Your job is to design and implement a centralized analytics warehouse in Amazon Redshift.

## Tasks Overview

1. **Explore and Plan** - Review CSV data, identify key fields, map to warehouse schema
2. **Design the Schema** - Create dimensional model with staging, dimension, and fact tables
3. **Extract and Transform** - Load source systems and extract/transform data
4. **Load into Redshift** - Execute DDL, load staging tables, populate dimensions and facts
5. **Optimize Performance** - Apply best practices, create materialized views
6. **Validate and Report** - Run quality checks, generate final report

---
## The Data

The dataset consists of three CSV files representing data from different operational systems:

### Orders Data (PostgreSQL source) - `ecom_orders_postgres.csv`
- **order_id**: Unique identifier for each order
- **customer_id**: Customer who placed the order
- **order_datetime / ship_datetime**: Timestamps for order and shipping
- **channel / device_type / browser**: How the order was placed
- **country / state**: Geographic location
- **payment_method / campaign**: Payment and marketing info
- **Financial fields**: subtotal, discount, shipping, tax, total amounts
- **Delivery fields**: delivery_days, on_time_delivery
- **Flags**: authorization_approved, returned

### Events Data (Cassandra source) - `ecom_events_cassandra.csv`
- **event_id / session_id**: Event and session identifiers
- **customer_id**: Customer who triggered the event
- **event_type**: Type of event (page_view, product_view, add_to_cart, etc.)
- **event_ts**: Timestamp of the event
- **Device/browser/OS info**: Technical context
- **Behavioral fields**: page_depth, latency_ms, dwell_seconds
- **Commerce fields**: cart_value_usd, discount_rate, fraud_score

### Graph Edges Data (Neo4j source) - `ecom_graph_edges_neo4j.csv`
- **edge_id**: Unique relationship identifier
- **from_node_id / to_node_id**: Source and target nodes
- **from_node_type / to_node_type**: Node types (Customer, Product, Order)
- **relationship**: Type of relationship (PURCHASED, VIEWED, ADDED_TO_CART, etc.)
- **Context fields**: order_id, category, customer_segment, campaign
- **Metrics**: edge_strength, unit_price_usd, quantity

---
## Setup: Imports and Dependencies

Run this cell first to import all required libraries.

In [2]:
# ========= Imports
import os, io, re, time, json, textwrap
from datetime import datetime
from typing import Dict, Any, List, Tuple
import numpy as np
import pandas as pd

# Source system libraries
import psycopg2
from psycopg2.extras import execute_values
from sqlalchemy import create_engine
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from neo4j import GraphDatabase

# Warehouse (Redshift) via Data API
import boto3

# Optional progress bars
try:
    from tqdm import tqdm
    TQDM = True
except Exception:
    TQDM = False

print("All imports successful!")
print(f"   - pandas version: {pd.__version__}")
print(f"   - numpy version: {np.__version__}")

All imports successful!
   - pandas version: 2.3.1
   - numpy version: 2.2.6


---
## Setup: Configuration

Update these settings for your environment. You will need to:
1. Set your AWS credentials (from Cloud Resources)
2. Configure database connection parameters

In [3]:
# Set up AWS credentials for the session (get these from Cloud Resources)
# IMPORTANT: Replace with your actual credentials

import os
os.environ["AWS_ACCESS_KEY_ID"] = "PASTE_YOUR_ACCESS_KEY_ID_HERE"
os.environ["AWS_SECRET_ACCESS_KEY"] = "PASTE_YOUR_SECRET_ACCESS_KEY_HERE"
os.environ["AWS_SESSION_TOKEN"] = "PASTE_YOUR_SESSION_TOKEN_HERE"

In [4]:
# ========= Configuration
BASE_DIR = os.getenv("PROJECT_BASE_DIR", ".")
DATA_DIR = os.path.join(BASE_DIR, "data")
CSV_ORDERS  = os.path.join(DATA_DIR, "ecom_orders_postgres.csv")
CSV_EVENTS  = os.path.join(DATA_DIR, "ecom_events_cassandra.csv")
CSV_EDGES   = os.path.join(DATA_DIR, "ecom_graph_edges_neo4j.csv")
DDL_MD_PATH = os.path.join(BASE_DIR, "project-ddl-long.md")
MERMAID_MD  = os.path.join(BASE_DIR, "project-mermaid-diagram.md")
BATCH_SIZE  = int(os.getenv("BATCH_SIZE", "1000"))

# PostgreSQL
PG_HOST = os.getenv("PG_HOST", "localhost")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DB   = os.getenv("PG_DB",   "postgres")
PG_USER = os.getenv("PG_USER", "temp")
PG_PW   = os.getenv("PG_PW",   "temp")

# Cassandra
CAS_HOSTS = os.getenv("CAS_HOSTS", "localhost").split(",")
CAS_PORT  = int(os.getenv("CAS_PORT", "9042"))
CAS_USER  = os.getenv("CAS_USER", "")
CAS_PW    = os.getenv("CAS_PW", "")
CAS_KEYSPACE = os.getenv("CAS_KEYSPACE", "ecommerce")

# Neo4j
NEO4J_URI  = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PW   = os.getenv("NEO4J_PW",   "neo4jpass")

# AWS/Redshift
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
REDSHIFT_DATABASE = os.getenv("REDSHIFT_DATABASE", "ecom")
REDSHIFT_WORKGROUP = os.getenv("REDSHIFT_WORKGROUP", "udacity-dwh-wg")
REDSHIFT_SECRET_ARN = os.getenv("REDSHIFT_SECRET_ARN")
REDSHIFT_CLUSTER_IDENTIFIER = os.getenv("REDSHIFT_CLUSTER_IDENTIFIER")
REDSHIFT_DB_USER = os.getenv("REDSHIFT_DB_USER")

# Verify configuration
print("Configuration loaded!")
print(f"   - BASE_DIR: {BASE_DIR}")
print(f"   - PostgreSQL: {PG_HOST}:{PG_PORT}/{PG_DB}")
print(f"   - Cassandra: {CAS_HOSTS}:{CAS_PORT}/{CAS_KEYSPACE}")
print(f"   - Neo4j: {NEO4J_URI}")
print(f"   - Redshift: {REDSHIFT_DATABASE} (workgroup: {REDSHIFT_WORKGROUP})")

print()
if AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY:
    print(f"   AWS credentials found (Key ID: {AWS_ACCESS_KEY_ID[:10]}...)")
else:
    print("   WARNING: AWS credentials NOT FOUND - set them above!")

Configuration loaded!
   - BASE_DIR: .
   - PostgreSQL: localhost:5432/postgres
   - Cassandra: ['localhost']:9042/ecommerce
   - Neo4j: bolt://localhost:7687
   - Redshift: ecom (workgroup: udacity-dwh-wg)

   AWS credentials found (Key ID: ASIAWCRTEP...)


---
## Setup: Column Specifications

These define the mapping from source columns to Redshift staging tables.
Use these as a reference when building your transformation logic.

In [5]:
# Column specs for Redshift staging (name, kind)
# kind: 's' = string, 'ts' = timestamp, 'i' = integer, 'f' = float, 'b' = boolean

ORDERS_COLSPEC = [
    ('order_id','s'),('customer_id','s'),('order_datetime','ts'),('ship_datetime','ts'),
    ('channel','s'),('device_type','s'),('browser','s'),('country','s'),('state','s'),
    ('payment_method','s'),('campaign','s'),('primary_category','s'),('num_distinct_items','i'),
    ('subtotal_usd','f'),('discount_rate','f'),('discount_amount_usd','f'),('shipping_method','s'),
    ('shipping_cost_usd','f'),('tax_rate','f'),('tax_amount_usd','f'),('order_total_usd','f'),
    ('order_weight_kg','f'),('delivery_days','i'),('on_time_delivery','b'),
    ('authorization_approved','b'),('returned','b')
]

EVENTS_COLSPEC = [
    ('event_id','s'),('customer_id','s'),('session_id','s'),('event_type','s'),('event_ts','ts'),
    ('device_type','s'),('browser','s'),('os','s'),('referrer','s'),('country','s'),('state','s'),
    ('ab_variant','s'),('is_logged_in','b'),('page_depth','i'),('latency_ms','i'),
    ('dwell_seconds','i'),('cart_value_usd','f'),('discount_rate','f'),('fraud_score','f'),
    ('payment_outcome','s'),('sequence_num','i'),('product_id','s'),('category','s'),('promo_code','s')
]

EDGES_COLSPEC = [
    ('edge_id','s'),('from_node_id','s'),('from_node_type','s'),('to_node_id','s'),('to_node_type','s'),
    ('relationship','s'),('timestamp','ts'),('order_id','s'),('category','s'),('customer_segment','s'),
    ('edge_strength','f'),('price_bucket','s'),('region','s'),('state','s'),('campaign','s'),
    ('same_household','b'),('prior_interactions','i'),('dwell_seconds','i'),('product_id','s'),
    ('unit_price_usd','f'),('quantity','i'),('returned_flag','b'),('auth_approved','b')
]

print(f"Column specs defined:")
print(f"   - Orders: {len(ORDERS_COLSPEC)} columns")
print(f"   - Events: {len(EVENTS_COLSPEC)} columns")
print(f"   - Edges: {len(EDGES_COLSPEC)} columns")

Column specs defined:
   - Orders: 26 columns
   - Events: 24 columns
   - Edges: 23 columns


---
## Setup: Helper Functions

Utility functions used throughout the pipeline.

In [6]:
# ========= Imports
import os, io, re, time, json, textwrap
from datetime import datetime
from typing import Dict, Any, List, Tuple
import numpy as np
import pandas as pd

def trim_df(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize text fields and handle NaN values."""
    df = df.copy()
    for c in df.select_dtypes(include=['object']).columns:
        df[c] = df[c].astype(str).str.strip()
        df[c] = df[c].replace({'nan': np.nan, 'None': np.nan, 'NaN': np.nan, '': np.nan})
    return df

def read_csvs() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Read all three source CSVs and apply cleaning."""
    orders = pd.read_csv(CSV_ORDERS)
    events = pd.read_csv(CSV_EVENTS)
    edges  = pd.read_csv(CSV_EDGES)
    return trim_df(orders), trim_df(events), trim_df(edges)

print("Helper functions defined: trim_df(), read_csvs()")

Helper functions defined: trim_df(), read_csvs()


---
# Task 1: Explore and Plan the Data Pipeline

In this task, you will:
- Review the provided CSV data files to understand structure, columns, and content
- Identify key fields and relationships important for analysis
- Map source fields to fact and dimension tables
- Consider data format standardization needs

**Deliverables:**
- Written plan mapping fields from all three sources to fact and dimension tables
- Documentation of key relationships and ID standardization strategies

In [7]:
# Read the CSV files
print("Reading CSV files...")
orders_df, events_df, edges_df = read_csvs()

print("\n" + "="*60)
print("TASK 1: Data Exploration")
print("="*60)

# TODO: Explore the orders data
# Display shape, columns, and sample rows
print(f"\n📊 ORDERS DATA (from PostgreSQL)")
print(f"   Shape: {orders_df.shape[0]} rows, {orders_df.shape[1]} columns")


# TODO: Print columns and display sample rows
print(f" Columns")


#Lets loop print the columns with the starting order as 1 using enumerate
for i, col in enumerate(orders_df.columns,1):
    print(f" {i}. {col}")
print(f" Sample Rows")


#Lets print some sample rows
print(orders_df.head())

Reading CSV files...

TASK 1: Data Exploration

📊 ORDERS DATA (from PostgreSQL)
   Shape: 2500 rows, 26 columns
 Columns
 1. order_id
 2. customer_id
 3. order_datetime
 4. ship_datetime
 5. channel
 6. device_type
 7. browser
 8. country
 9. state
 10. payment_method
 11. campaign
 12. primary_category
 13. num_distinct_items
 14. subtotal_usd
 15. discount_rate
 16. discount_amount_usd
 17. shipping_method
 18. shipping_cost_usd
 19. tax_rate
 20. tax_amount_usd
 21. order_total_usd
 22. order_weight_kg
 23. delivery_days
 24. on_time_delivery
 25. authorization_approved
 26. returned
 Sample Rows
    order_id customer_id       order_datetime        ship_datetime  \
0  ORD100000      C29457  2024-06-12 02:14:21  2024-06-17 02:14:21   
1  ORD100001      C22666  2024-08-08 00:16:41  2024-08-11 00:16:41   
2  ORD100002      C72623  2024-09-11 06:59:18  2024-09-15 06:59:18   
3  ORD100003      C62733  2025-03-13 15:54:14  2025-03-17 15:54:14   
4  ORD100004      C62083  2025-02-28 00:20:

In [8]:
# TODO: Explore the events data
print(f"\n📊 EVENTS DATA (from Cassandra)")
print(f"   Shape: {events_df.shape[0]} rows, {events_df.shape[1]} columns")
# TODO: Print columns and display sample rows

# Lets print the columns using the enumerate with starting order as 1
print("Sample columns")
for i, col in enumerate(events_df.columns,1):
    print(f" {i}. {col}")

# Lets print some rows
print("Sample rows")
print(events_df.head())




📊 EVENTS DATA (from Cassandra)
   Shape: 2500 rows, 24 columns
Sample columns
 1. event_id
 2. customer_id
 3. session_id
 4. event_type
 5. event_ts
 6. device_type
 7. browser
 8. os
 9. referrer
 10. country
 11. state
 12. ab_variant
 13. is_logged_in
 14. page_depth
 15. latency_ms
 16. dwell_seconds
 17. cart_value_usd
 18. discount_rate
 19. fraud_score
 20. payment_outcome
 21. sequence_num
 22. product_id
 23. category
 24. promo_code
Sample rows
    event_id customer_id   session_id    event_type             event_ts  \
0  EVT200000      C47857  S7645111197     page_view  2024-11-14 00:17:09   
1  EVT200001      C83195  S1423573902     page_view  2024-08-15 10:42:18   
2  EVT200002      C27664  S2829853742  product_view  2024-05-03 23:28:24   
3  EVT200003      C55911  S2500554740   add_to_cart  2024-10-28 18:44:50   
4  EVT200004      C27347  S3072616664   add_to_cart  2024-07-22 07:43:54   

  device_type  browser       os        referrer country  ... latency_ms  \
0     d

In [9]:
# TODO: Explore the graph edges data
print(f"\n📊 GRAPH EDGES DATA (from Neo4j)")
print(f"   Shape: {edges_df.shape[0]} rows, {edges_df.shape[1]} columns")
# TODO: Print columns and display sample rows

# Lets print the columns using the enumerate with starting order as 1
print("Sample columns")
for i, col in enumerate(edges_df.columns,1):
    print(f" {i}. {col}")

# Lets print some rows
print("Sample rows")
print(edges_df.head())



📊 GRAPH EDGES DATA (from Neo4j)
   Shape: 2500 rows, 23 columns
Sample columns
 1. edge_id
 2. from_node_id
 3. from_node_type
 4. to_node_id
 5. to_node_type
 6. relationship
 7. timestamp
 8. order_id
 9. category
 10. customer_segment
 11. edge_strength
 12. price_bucket
 13. region
 14. state
 15. campaign
 16. same_household
 17. prior_interactions
 18. dwell_seconds
 19. product_id
 20. unit_price_usd
 21. quantity
 22. returned_flag
 23. auth_approved
Sample rows
      edge_id from_node_id from_node_type to_node_id to_node_type  \
0  EDGE300000       C65482       Customer      P2261      Product   
1  EDGE300001       C59599       Customer      P7625      Product   
2  EDGE300002       C44719       Customer      P8677      Product   
3  EDGE300003       C13897       Customer      P7883      Product   
4  EDGE300004       C30350       Customer      P2015      Product   

    relationship            timestamp   order_id category customer_segment  \
0        RETURNS  2024-01-27 13

In [10]:
# TODO: Identify key fields and relationships
print("\n" + "="*60)
print("KEY FIELDS AND RELATIONSHIPS")
print("="*60)

# TODO: Document the following:
# 1. Primary keys for each data source
# 2. Foreign key relationships between sources
# 3. Date/time fields that will need date dimension lookups
# 4. Categorical fields that should become dimensions

print("\n🔑 Primary Keys:")
# TODO: Print unique counts for primary key fields

# Based on the rows that we printed previously, 
# we can find out that order_id, event_id, edge_id are the primary keys

print(f" order_id:  {orders_df['order_id'].nunique()} unique, {len(orders_df)} rows")

print(f" event_id:  {events_df['event_id'].nunique()} unique, {len(events_df)} rows")

print(f" edge_id:  {edges_df['edge_id'].nunique()} unique, {len(edges_df)} rows")


print("\n🔗 Foreign Key Relationships:")
# TODO: Document how sources relate to each other

print("orders_df:")
print("order_id is the primary key.")
print("customer_id - conforming ID as I am seeing this in the events.customer_id as well")


print("events_df:")
print("event_id is the primary key")
print("customer_id - conforming ID with orders.customer_id")
print("product_id - conforming ID with edges.product_id ")


print("edges_df:")
print("edge_id is the primary key")
print("order_id is the foreign key and relates to orders_df")
print("product_id - conforming ID with events.product_id")
# the below is purely based on the future relationships. Keeping it for reference 
print("from_node_id or to_node_id - likely customer_id or product_id, depending on from_node_type or to_node_type")

print("\n📅 Date/Time Fields:")
# TODO: List timestamp fields from each source
print("orders_df- order_datetime, ship_datetime")
print("events_df- event_ts")
print("edges_df- timestamp")



print("\n📋 Categorical Fields (potential dimensions):")
# TODO: List categorical fields and their cardinality
# To quickly identify the categorical fields, lets start by identifying the 
# string kind columns from COLSPEC and drop the ones that are IDs.
# High Cardinality is almost like a ID field. So, we have to drop those kind of fields

# Lets assign the list var to the categorical fields
categorical_orders = ['channel', 'device_type', 'browser', 'country', 'state', 'payment_method',
                       'campaign', 'primary_category', 'shipping_method']
categorical_events = ['event_type', 'device_type', 'browser', 'os', 'referrer', 'country', 'state',
                       'ab_variant', 'payment_outcome', 'category', 'promo_code']
categorical_edges = ['from_node_type', 'to_node_type', 'relationship', 'category', 'customer_segment',
                      'price_bucket', 'region', 'state', 'campaign']

print("orders_df categorical fields:")
for col in categorical_orders:
    print(f"{col} - {orders_df[col].nunique()} unique values")

print("events_df categorical fields:")
for col in categorical_events:
    print(f"{col} - {events_df[col].nunique()} unique values")
        
print("edges_df categorical fields:")
for col in categorical_edges:
    print(f"{col} - {edges_df[col].nunique()} unique values")


KEY FIELDS AND RELATIONSHIPS

🔑 Primary Keys:
 order_id:  2500 unique, 2500 rows
 event_id:  2500 unique, 2500 rows
 edge_id:  2500 unique, 2500 rows

🔗 Foreign Key Relationships:
orders_df:
order_id is the primary key.
customer_id - conforming ID as I am seeing this in the events.customer_id as well
events_df:
event_id is the primary key
customer_id - conforming ID with orders.customer_id
product_id - conforming ID with edges.product_id 
edges_df:
edge_id is the primary key
order_id is the foreign key and relates to orders_df
product_id - conforming ID with events.product_id
from_node_id or to_node_id - likely customer_id or product_id, depending on from_node_type or to_node_type

📅 Date/Time Fields:
orders_df- order_datetime, ship_datetime
events_df- event_ts
edges_df- timestamp

📋 Categorical Fields (potential dimensions):
orders_df categorical fields:
channel - 5 unique values
device_type - 3 unique values
browser - 5 unique values
country - 6 unique values
state - 51 unique value

In [11]:
# TODO: Field mapping - source and dimensional models

# DDL's own 'source data mapping' table tells us which staging table (and csv) feeds which fact table:
# ecom_orders_postgres.csv      -> stg.orders_raw  -> dw.fact_orders
# ecom_events_cassandra.csv     -> stg.events_raw  -> dw.fact_events
# ecom_graph_edges_neo4j.csv    -> stg.edges_raw   -> dw.fact_graph_edges

# Below, each source is compared against its relevant fact table's DDL columns
# Based on the analysis, the below are possible outcomes:
# a. Same name appears directly - easy map.
# b. "_sk" version appears - needs a dim_* lookup first
# c. Neither - flagged as not used


print("FIELD MAPPING: source columns -> warehouse tables")

orders_mapping = {
    # Based on analysis 'a'
    'order_id':                'fact_orders.order_id (business key)',
    # Based on analysis 'b'
    'customer_id':              'dim_customer lookup -> fact_orders.customer_sk',
    'order_datetime':           'date part -> dim_date.date_key -> fact_orders.order_date_key',
    'ship_datetime':            'date part -> dim_date.date_key -> fact_orders.ship_date_key',
    'channel':                  'dim_channel.channel -> fact_orders.channel_sk',
    'device_type':              'dim_device.device_type -> fact_orders.device_sk',
    'browser':                  'dim_browser.browser -> fact_orders.browser_sk',
    # Based on analysis 'c'
    # country/state exist on stg.orders_raw, but fact_orders has no such columns -
    # they only show up on dim_customer, so they describe the customer, not the order
    'country':                  'dim_customer.country (customer attribute, not on fact_orders)',
    'state':                    'dim_customer.state (customer attribute, not on fact_orders)',
    #Based on analysis 'a'
    'payment_method':           'dim_payment_method.payment_method -> fact_orders.payment_method_sk',
    'campaign':                 'dim_campaign.campaign -> fact_orders.campaign_sk',
    'primary_category':         'fact_orders.primary_category (direct, not dimensionalized)',
    'num_distinct_items':       'fact_orders.num_distinct_items (measure)',
    'subtotal_usd':             'fact_orders.subtotal_usd (measure)',
    'discount_rate':            'fact_orders.discount_rate (measure)',
    'discount_amount_usd':      'fact_orders.discount_amount_usd (measure)',
    'shipping_method':          'dim_shipping_method.shipping_method -> fact_orders.shipping_method_sk',
    'shipping_cost_usd':        'fact_orders.shipping_cost_usd (measure)',
    'tax_rate':                 'fact_orders.tax_rate (measure)',
    'tax_amount_usd':           'fact_orders.tax_amount_usd (measure)',
    'order_total_usd':          'fact_orders.order_total_usd (measure)',
    'order_weight_kg':          'fact_orders.order_weight_kg (measure)',
    'delivery_days':            'fact_orders.delivery_days (measure)',
    'on_time_delivery':         'fact_orders.on_time_delivery (flag)',
    'authorization_approved':   'fact_orders.authorization_approved (flag)',
    'returned':                 'fact_orders.returned (flag)',
}

print("orders_df:")
for col, dest in orders_mapping.items():
    print(f"  {col:24} -> {dest}")

events_mapping = {
    'event_id':          'fact_events.event_id (business key)',                          # a
    'customer_id':       'dim_customer lookup -> fact_events.customer_sk',               # b
    'session_id':        'fact_events.session_id (degenerate dimension, kept as-is)',    # a
    'event_type':        'fact_events.event_type (direct)',                              # a
    'event_ts':          'date part -> dim_date.date_key -> fact_events.event_date_key', # b
    'device_type':       'dim_device.device_type -> fact_events.device_sk',              # b
    'browser':           'dim_browser.browser -> fact_events.browser_sk',                # b
    'os':                'dim_os.os -> fact_events.os_sk',                               # b
    'referrer':          'dim_referrer.referrer -> fact_events.referrer_sk',             # b
    # same reasoning as orders_df.country/state above - only dim_customer carries these
    'country':           'not used - only dim_customer carries country',                 # c
    'state':             'not used - only dim_customer carries state',                   # c
    'ab_variant':        'dim_ab_variant.ab_variant -> fact_events.ab_variant_sk',        # b
    'is_logged_in':      'dim_customer.is_logged_in (customer attribute, not on fact_events)', # c
    'page_depth':        'fact_events.page_depth (measure)',                             # a
    'latency_ms':        'fact_events.latency_ms (measure)',                             # a
    'dwell_seconds':     'fact_events.dwell_seconds (measure)',                          # a
    'cart_value_usd':    'fact_events.cart_value_usd (measure)',                         # a
    'discount_rate':     'fact_events.discount_rate (measure)',                          # a
    'fraud_score':       'fact_events.fraud_score (measure)',                            # a
    'payment_outcome':   'fact_events.payment_outcome (direct)',                         # a
    'sequence_num':      'fact_events.sequence_num (measure)',                           # a
    'product_id':        'dim_product lookup -> fact_events.product_sk',                 # b
    'category':          'fact_events.category (direct)',                               # a
    'promo_code':        'fact_events.promo_code (direct)',                              # a
}

print("events_df:")
for col, dest in events_mapping.items():
    print(f"  {col:24} -> {dest}")

# fact_events has a channel_sk column in the DDL, but stg.events_raw / events_df
# has no "channel" field to fill it from - flagging this as a gap in the provided
# DDL rather than silently leaving it out of the mapping
print("NOTE: fact_events also has a channel_sk column, but events_df has no 'channel' source field - nothing maps to it.")

edges_mapping = {
    'edge_id':             'fact_graph_edges.edge_id (business key)',   # a

    # from_node_id/to_node_id are generic (could be a Customer or a Product) -
    # from_node_type/to_node_type tell us which one, so the actual lookup target
    # depends on that type, not just the id alone
    'from_node_id/type':  'resolved by node type -> from_customer_sk OR from_product_sk',  # b
    'to_node_id/type':    'resolved by node type -> to_customer_sk OR to_product_sk',      # b
    'relationship':       'fact_graph_edges.relationship (direct)',    # a
    'timestamp':           'date part -> dim_date.date_key -> fact_graph_edges.event_date_key',  # b
    'order_id':            'fact_graph_edges.order_id (direct, nullable for non-order edges)',    # a
    'category':          'fact_graph_edges.category (direct, not dimensionalized)',    # a
    'customer_segment':    'fact_graph_edges.customer_segment (direct, not dimensionalized)',  # a
    'edge_strength':     'fact_graph_edges.edge_strength (measure)',  # a
    'price_bucket':        'fact_graph_edges.price_bucket (direct)',    # a
    'region':              'fact_graph_edges.region (direct)',          # a
    'state':               'fact_graph_edges.state (direct)',           # a
    'campaign':            'dim_campaign lookup -> fact_graph_edges.campaign_sk',  # b

    # same_household exists on stg.edges_raw but nowhere on fact_graph_edges -
    # another apparent gap in the provided DDL, same as channel_sk in events_mapping
    'same_household':      'not used - no matching column in fact_graph_edges',    # c
    'prior_interactions': 'fact_graph_edges.prior_interactions (measure)',  # a
    'dwell_seconds':       'fact_graph_edges.dwell_seconds (measure)',       # a

    # product_id on its own isn't carried through - the from/to_product_sk pair
    # (resolved above, from from_node_id/to_node_id) already captures which product(s) are involved
    'product_id':          'superseded by from_product_sk/to_product_sk resolution',  # c
    'unit_price_usd':      'fact_graph_edges.unit_price_usd (measure)',  # a
    'quantity':            'fact_graph_edges.quantity (measure)',        # a
    'returned_flag':       'fact_graph_edges.returned_flag (flag)',      # a
    'auth_approved':       'fact_graph_edges.auth_approved (flag)',      # a
}


print("\nedges_df ->")
for col, dest in edges_mapping.items():
    print(f"  {col:24} -> {dest}")

# ID standardization: checking whether the same conceptual ID (customer_id,
# product_id) needs any reformatting before it can be used to join across
# sources. Since all three sources were generated for this project with the
# same VARCHAR(32) id format, no cleanup step is needed here - but this is the
# kind of check that would matter a lot with real, independently-built systems.

print("ID STANDARDIZATION")

print("customer_id - same VARCHAR(32) format in orders_df and events_df, no reformatting needed")
print("product_id  - same VARCHAR(32) format in events_df and edges_df, no reformatting needed")
print("from_node_id/to_node_id (edges_df) - VARCHAR(32), matches customer_id/product_id format,")
print("            but needs from_node_type/to_node_type to know which dimension to resolve against")

FIELD MAPPING: source columns -> warehouse tables
orders_df:
  order_id                 -> fact_orders.order_id (business key)
  customer_id              -> dim_customer lookup -> fact_orders.customer_sk
  order_datetime           -> date part -> dim_date.date_key -> fact_orders.order_date_key
  ship_datetime            -> date part -> dim_date.date_key -> fact_orders.ship_date_key
  channel                  -> dim_channel.channel -> fact_orders.channel_sk
  device_type              -> dim_device.device_type -> fact_orders.device_sk
  browser                  -> dim_browser.browser -> fact_orders.browser_sk
  country                  -> dim_customer.country (customer attribute, not on fact_orders)
  state                    -> dim_customer.state (customer attribute, not on fact_orders)
  payment_method           -> dim_payment_method.payment_method -> fact_orders.payment_method_sk
  campaign                 -> dim_campaign.campaign -> fact_orders.campaign_sk
  primary_category         

In [12]:
# TODO: Document your field mappings and data quality observations
print("\n" + "="*60)
print("DATA QUALITY SUMMARY")
print("="*60)

# TODO: Check for null values in key columns
# TODO: Document any data quality issues found

# Based on the primary key, conforming/Foreign Key identified before, 
# lets see whether those fields have any nulls

print("orders_df:")
for col in ['order_id', 'customer_id', 'order_datetime', 'ship_datetime']:
    print(f"{col} - {orders_df[col].isna().sum()} nulls")

print("events_df:")
for col in ['event_id', 'customer_id', 'product_id', 'event_ts']:
    print(f"{col} - {events_df[col].isna().sum()} nulls")

print("edges_df:")
for col in ['edge_id', 'order_id', 'product_id', 'from_node_id', 'to_node_id', 'timestamp']:
    print(f"{col} - {edges_df[col].isna().sum()} nulls")
    
    
print("Data Quality Findings:")
print("No nulls found in primary keys - orders, events, edges")
print("events_df has 1118 nulls for product_id")
print("edges_df has 1527 nulls for order_id")
print("Lets see why:")
print()
print("events_df.product_id — breakdown by event_type:")

# We have to first identify the null pattern. 
# Lets see what event_type have nulls for the product id and how much each
null_by_event_type = events_df.groupby('event_type')['product_id'].apply(lambda s: s.isna().sum())
print(null_by_event_type)

# types - page_view, checkout_start and payment_attemp have nulls
# Lets create a dictionary for the identified types and assign a descriptive value
event_notes = {
    'page_view':       'general browsing — not about one specific product',
    'checkout_start':  'cart-level action — not tied to one product',
    'payment_attempt': 'order-level action — not tied to one product',
}

# for every event type and its count, lets add a descriptive note based on the above dictionary values. 
# If no values found, meaning, there are no nulls, then lets add a default message - expected to always have a product_id
for event_type, null_count in null_by_event_type.items():
    note = event_notes.get(event_type, 'expected to always have a product_id')
    print(f"   {event_type:18} {null_count:4} nulls   ({note})")

print()
print("edges_df.order_id — breakdown by relationship:")

# We have to first identify the null pattern. 
# Lets see what relationship have nulls for the order id and how much each
null_by_relationship = edges_df.groupby('relationship')['order_id'].apply(lambda s: s.isna().sum())
print(null_by_relationship)

# types - viewed, added_to_cart and referred_friend have nulls
# Lets create a dictionary for the identified types and assign a descriptive value
relationship_notes = {
    'VIEWED':           'browsing behavior — not tied to any order',
    'ADDED_TO_CART':    'cart action — not tied to a completed order',
    'REFERRED_FRIEND':  'customer-to-customer — not tied to any order',
}

# for every relationship type and its count, lets add a descriptive note based on the above dictionary values. 
# If no values found, meaning, there are no nulls, then lets add a default message - expected to always have a product_id
for relationship, null_count in null_by_relationship.items():
    note = relationship_notes.get(relationship, 'expected to always have an order_id')
    print(f"   {relationship:18} {null_count:4} nulls   ({note})")

print()
print("Conclusion: both null patterns are fully explained by category.")
print("Every null falls into an event_type/relationship that logically wouldn't carry that value.")
print("Not a data quality issue. Documented as expected structural behavior.")


DATA QUALITY SUMMARY
orders_df:
order_id - 0 nulls
customer_id - 0 nulls
order_datetime - 0 nulls
ship_datetime - 0 nulls
events_df:
event_id - 0 nulls
customer_id - 0 nulls
product_id - 1118 nulls
event_ts - 0 nulls
edges_df:
edge_id - 0 nulls
order_id - 1527 nulls
product_id - 0 nulls
from_node_id - 0 nulls
to_node_id - 0 nulls
timestamp - 0 nulls
Data Quality Findings:
No nulls found in primary keys - orders, events, edges
events_df has 1118 nulls for product_id
edges_df has 1527 nulls for order_id
Lets see why:

events_df.product_id — breakdown by event_type:
event_type
add_to_cart           0
checkout_start      259
page_view           641
payment_attempt     218
product_view          0
purchase              0
return_initiated      0
Name: product_id, dtype: int64
   add_to_cart           0 nulls   (expected to always have a product_id)
   checkout_start      259 nulls   (cart-level action — not tied to one product)
   page_view           641 nulls   (general browsing — not about

---
# Task 2: Design the Warehouse Schema

In this task, you will:
- Review the dimensional (star) schema design
- Understand staging tables, dimension tables, and fact tables
- Review distribution keys, sort keys, and encoding for optimization
- Document the purpose of each table

The DDL is defined in `project-ddl-long.md`. Review the schema design and understand how it supports analytics.

**Deliverables:**
- Understanding of the provided DDL structure
- Documentation of table purposes and query support

In [13]:
# Review the DDL file
print("="*60)
print("TASK 2: Warehouse Schema Design")
print("="*60)

print("\n📄 Reading DDL file:", DDL_MD_PATH)

# TODO: Read and parse the DDL file
with open(DDL_MD_PATH, 'r') as f:
    ddl_content = f.read()

# TODO: Count and list the tables defined
# - How many staging tables?
# - How many dimension tables?
# - How many fact tables?

# re lets us search (regular expression) text using a pattern
import re


# scan the whole DDL file's text for every "CREATE TABLE <name>" and pull out just the names
# result is a plain list of table names
table_names = re.findall(r'CREATE TABLE (\S+)', ddl_content)

# lets start with three empty lists - we will sort each table name into the right one below
staging_tables = []
dim_tables = []
fact_tables = []

# lets go through every table name once, and decide which list it belongs in
# stg.  -> staging table
# .dim_ -> dimension table
# .fact_ -> fact table

for t in table_names:
    if t.startswith('stg'):
        staging_tables.append(t)
    elif '.dim_' in t:
        dim_tables.append(t)
    elif '.fact_' in t:
        fact_tables.append(t)
        
# print each group with its count, then list every table name in that group
print(f"Staging tables {len(staging_tables)}")
for t in staging_tables:
    print(t)

print(f"Dimension tables {len(dim_tables)}")
for t in dim_tables:
    print(t)

print(f"Fact tables {len(fact_tables)}")
for t in fact_tables:
    print(t)

    

TASK 2: Warehouse Schema Design

📄 Reading DDL file: ./project-ddl-long.md
Staging tables 3
stg.orders_raw
stg.events_raw
stg.edges_raw
Dimension tables 12
dw.dim_date
dw.dim_customer
dw.dim_product
dw.dim_campaign
dw.dim_channel
dw.dim_device
dw.dim_browser
dw.dim_os
dw.dim_referrer
dw.dim_shipping_method
dw.dim_payment_method
dw.dim_ab_variant
Fact tables 3
dw.fact_orders
dw.fact_events
dw.fact_graph_edges


In [14]:
# TODO: Document the schema design
print("\n" + "="*60)
print("SCHEMA DESIGN DOCUMENTATION")
print("="*60)

# TODO: Document the purpose of each table type and key design decisions
# Consider:
# - Why are staging tables needed?
# - What dimensions are being used?
# - What is the grain of each fact table?
# - Why were specific DISTKEY and SORTKEY choices made?

schema_doc = """
## Staging Tables (stg schema)
# TODO: List and describe staging tables
stg.orders_raw: raw landing zone for ecom_orders_postgres.csv 
stg.events_raw: raw landing zone for ecom_events_cassandra.csv 
stg.edges_raw: raw landing zone for ecom_graph_edges_neo4j.csv 
Staging allows messy data to be stored safely, so cleaning and transforming is a separate step and we 
dont have to pull from the source each time.

## Dimension Tables (dw schema)
# TODO: List and describe dimension tables
dw.dim_date: calendar dimension, one row per day, used by every fact table for date lookups
dw.dim_customer: one row per customer, slowly changing dimension Type 2 (tracks changes over time)
dw.dim_product: one row per product, slowly changining dimension Type 2
dw.dim_campaign: marketing campaign names
dw.dim_channel: order channel (web, mobile_app, etc.)
dw.dim_device: device type (desktop, mobile, tablet)
dw.dim_browser: browser name
dw.dim_os: operating system
dw.dim_referrer: traffic referrer source
dw.dim_shipping_method: shipping method
dw.dim_payment_method: payment method
dw.dim_ab_variant: A/B test variant
12 dimension tables total - 3 (date, customer, product) track more detail/history, and 9 are small fixed
lookup lists.


## Fact Tables (dw schema)
# TODO: List and describe fact tables
dw.fact_orders: one row per order, built from stg.orders_raw
dw.fact_events: one row per event, built from stg.events_raw
dw.fact_graph_edges: one row per graph edge, built from stg.edges_raw
Each fact table matches one source system's natural grain - an order, an event, or a graph edge
so no single fact table mixes two different kinds of occurrence together.

## Optimization Choices
# TODO: Document DISTKEY, SORTKEY, and encoding decisions

DISTKEY(customer_sk) on fact_orders and fact_events, colocates fact_orders and fact_events with each other 
(both use the same DISTKEY), and speeds up GROUP BY customer_sk aggregation within a
single fact table. Note: dim_customer's own DISTKEY is actually customer_id, not customer_sk, so this does not 
guarantee a shuffle-free join straight to dim_customer, despite that being the usual
textbook reason given for this choice.

DISTKEY(to_product_sk) on fact_graph_edges, same idea, centered on products instead of customers.
Same note applies: dim_product's own DISTKEY is product_id, not product_sk.

DISTSTYLE ALL on the 9 small dimensions (campaign, channel, device, browser, os, referrer, shipping_method,
payment_metohd, ab_variant): a full copy sits on every node, so joining against them
never needs a shuffle either.

SORTKEY on the date key for every fact table, lets Redshift skip blocks of data when a query
filters by date range, instead of scanning the whole table.

ENCODE zstd on most columns: reduces storage size and speeds up scans.

"""

print(schema_doc)



SCHEMA DESIGN DOCUMENTATION

## Staging Tables (stg schema)
# TODO: List and describe staging tables
stg.orders_raw: raw landing zone for ecom_orders_postgres.csv 
stg.events_raw: raw landing zone for ecom_events_cassandra.csv 
stg.edges_raw: raw landing zone for ecom_graph_edges_neo4j.csv 
Staging allows messy data to be stored safely, so cleaning and transforming is a separate step and we 
dont have to pull from the source each time.

## Dimension Tables (dw schema)
# TODO: List and describe dimension tables
dw.dim_date: calendar dimension, one row per day, used by every fact table for date lookups
dw.dim_customer: one row per customer, slowly changing dimension Type 2 (tracks changes over time)
dw.dim_product: one row per product, slowly changining dimension Type 2
dw.dim_campaign: marketing campaign names
dw.dim_channel: order channel (web, mobile_app, etc.)
dw.dim_device: device type (desktop, mobile, tablet)
dw.dim_browser: browser name
dw.dim_os: operating system
dw.dim_referre

---
# Task 3: Extract and Transform the Source Data

In this task, you will:
- Load CSV data into PostgreSQL, Cassandra, and Neo4j (simulating production)
- Write extraction functions to query each source system
- Apply transformations to clean and conform the data
- Ensure transformed data matches staging table specifications

**Deliverables:**
- Working functions to connect, load, and extract from each source
- Transformed DataFrames ready for Redshift loading

## Task 3.1: Define Source System Functions

Implement the connection and data loading functions for each source system.

In [15]:
# ========= PostgreSQL Functions =========

def pg_connect():
    """Connect to PostgreSQL."""
    # TODO: Implement using psycopg2.connect()
    # Use: PG_HOST, PG_PORT, PG_DB, PG_USER, PG_PW
    # The below returns a connection object and we will use that to run our queries later
    return psycopg2.connect(
        host=PG_HOST,
        port=PG_PORT,
        dbname=PG_DB,
        user=PG_USER,
        password=PG_PW
    )
    
def pg_load_orders(df: pd.DataFrame):
    """Create table and load orders data into PostgreSQL."""
    # TODO: Implement the following steps:
 
    conn=pg_connect()
    cur=conn.cursor() # cursor is required to execute commands and read results.
    
    # 1. Create schema: CREATE SCHEMA IF NOT EXISTS raw;
    cur.execute("CREATE SCHEMA IF NOT EXISTS raw;")    
    
    # 2. Create table: CREATE TABLE IF NOT EXISTS raw.orders (...)
    # the below mirrors the source system design
    cur.execute("""
        CREATE TABLE IF NOT EXISTS raw.orders (
            order_id VARCHAR(32),
            customer_id VARCHAR(32),
            order_datetime TIMESTAMP,
            ship_datetime TIMESTAMP,
            channel VARCHAR(32),
            device_type VARCHAR(16),
            browser VARCHAR(16),
            country VARCHAR(8),
            state VARCHAR(8),
            payment_method VARCHAR(16),
            campaign VARCHAR(32),
            primary_category VARCHAR(32),
            num_distinct_items INTEGER,
            subtotal_usd DECIMAL(12,2),
            discount_rate DECIMAL(5,3),
            discount_amount_usd DECIMAL(12,2),
            shipping_method VARCHAR(16),
            shipping_cost_usd DECIMAL(12,2),
            tax_rate DECIMAL(6,4),
            tax_amount_usd DECIMAL(12,2),
            order_total_usd DECIMAL(12,2),
            order_weight_kg DECIMAL(10,2),
            delivery_days INTEGER,
            on_time_delivery BOOLEAN,
            authorization_approved BOOLEAN,
            returned BOOLEAN
        );
    """)
    # good entry point to save what is done so far
    conn.commit()
    
    # lets makes this function safe to re-run - without this, calling it twice would
    # duplicate every row, since there's no PRIMARY KEY to upsert against
    cur.execute("TRUNCATE TABLE raw.orders;")
    conn.commit()
    
    # 3. Bulk insert using execute_values()
    # turn the DataFrame into a plain list of rows, then insert all of them in one call
    # execute_values is much faster than looping and running one INSERT per row
    rows = df.values.tolist()
    execute_values(cur, "INSERT INTO raw.orders VALUES %s", rows)
    conn.commit()
    cur.close()
    conn.close()
    print("Insertion of records complete")
    
def extract_from_pg() -> pd.DataFrame:
    """Extract orders from PostgreSQL."""
    # TODO: Implement using SQLAlchemy create_engine() and pd.read_sql_query()
    engine= create_engine(f"postgresql://{PG_USER}:{PG_PW}@{PG_HOST}:{PG_PORT}/{PG_DB}")
    # lets run the select query and return the dataframe result
    df = pd.read_sql_query("SELECT * FROM raw.orders", engine)
    engine.dispose()
    return df

print("PostgreSQL functions defined")

PostgreSQL functions defined


In [16]:
# ========= Cassandra Functions =========

def cas_connect():
    """Connect to Cassandra and ensure keyspace exists."""
    # TODO: Implement using cassandra.cluster.Cluster
    # 1. Create cluster connection (with auth if CAS_USER is set)
    # 2. Create keyspace if not exists
    # 3. Set keyspace and return session, cluster
    
    # if CAS_USER is set, lets connect with username/password; otherwise no auth needed
    if CAS_USER:
        auth_provider = PlainTextAuthProvider(username=CAS_USER, password=CAS_PW)
        cluster = Cluster(CAS_HOSTS, port=CAS_PORT, auth_provider=auth_provider)
    else:
        cluster= Cluster(CAS_HOSTS, port=CAS_PORT)
    
    # connect without picking a keyspace yet, the keypsace might not exist yet
    session=cluster.connect()
    
    # SimpleStrategy + replication factor 1 is the standard choice for a single-node setup
    session.execute(f"""
        CREATE KEYSPACE IF NOT EXISTS {CAS_KEYSPACE}
        WITH replication = {{'class': 'SimpleStrategy', 'replication_factor': 1}};
    """)

    # lets point this session at our keyspace for queries
    session.set_keyspace(CAS_KEYSPACE)
    return session, cluster
    
def cas_load_events(df: pd.DataFrame):
    """Create table and load events data into Cassandra."""
    # TODO: Implement the following steps:
    # 1. Connect to Cassandra
    # 2. Create events table
    # 3. Prepare INSERT statement
    # 4. Execute concurrent inserts using execute_concurrent_with_args
    from cassandra.concurrent import execute_concurrent_with_args
    
    session, cluster= cas_connect()
    
    # event_id as PRIMARY KEY since Task 1 confirmed it's unique. 
    session.execute("""
        CREATE TABLE IF NOT EXISTS events (
            event_id TEXT PRIMARY KEY,
            customer_id TEXT,
            session_id TEXT,
            event_type TEXT,
            event_ts TIMESTAMP,
            device_type TEXT,
            browser TEXT,
            os TEXT,
            referrer TEXT,
            country TEXT,
            state TEXT,
            ab_variant TEXT,
            is_logged_in BOOLEAN,
            page_depth INT,
            latency_ms INT,
            dwell_seconds INT,
            cart_value_usd DOUBLE,
            discount_rate DOUBLE,
            fraud_score DOUBLE,
            payment_outcome TEXT,
            sequence_num INT,
            product_id TEXT,
            category TEXT,
            promo_code TEXT
        );
    """)
    
    # prepared statement is a query template Cassandra parses once and reuses for every
    # row, instead of re-parsing the same INSERT text 2500 times. The ? marks are placeholders
    insert_stmt = session.prepare("""
        INSERT INTO events (
            event_id, customer_id, session_id, event_type, event_ts,
            device_type, browser, os, referrer, country, state,
            ab_variant, is_logged_in, page_depth, latency_ms, dwell_seconds,
            cart_value_usd, discount_rate, fraud_score, payment_outcome,
            sequence_num, product_id, category, promo_code
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """)
    
    # Cassandra needs a real datetime object for a TIMESTAMP column, not a string
    # pd.read_csv() doesn't parse dates automatically, so event_ts is still plain text
    # at this point (Postgres accepted the string fine since it casts server-side;
    # Cassandra's driver checks the Python type client-side before sending anything)
    
    df = df.copy()
    df['event_ts'] = pd.to_datetime(df['event_ts'])
    
    # Cassandra's driver only understands Python's None for a missing value, not
    # pandas' NaN product_id/category/payment_outcome all have real nulls here
    # (Task 1's finding: page_view/checkout_start/payment_attempt events genuinely
    # have no product_id), and NaN would crash the TEXT column serializer
    df = df.where(pd.notnull(df), None)
    
    # same idea as pg_load_orders - plain list of rows, one row per list item\
    rows= df.values.tolist()
    
    # lets run all inserts using execute concurrent with args
    execute_concurrent_with_args(session, insert_stmt, rows)
    
    cluster.shutdown()
    print("Inserted events in Cassandra")

def extract_from_cas() -> pd.DataFrame:
    """Extract events from Cassandra."""
    # TODO: Implement - query all events and return as DataFrame
    
    session, cluster = cas_connect()
    # lets use pandas dataframe from the list of rows using the row's own field names
    rows=session.execute("SELECT * FROM events")
    df=pd.DataFrame(list(rows))
    return df

print("Cassandra functions defined")

Cassandra functions defined


In [17]:
# ========= Neo4j Functions =========

def neo4j_driver():
    """Connect to Neo4j."""
    # TODO: Implement using GraphDatabase.driver()
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PW))

def neo4j_load_edges(df: pd.DataFrame):
    """Load edges into Neo4j as nodes and relationships."""
    # TODO: Implement the following steps:
    # 1. Clean and validate records
    # 2. Create node constraints
    # 3. Use MERGE to create nodes and relationships
    # 4. Process in batches
    
    driver=neo4j_driver()
    # Task 1 found 0 nulls in from_node_id/to_node_id/relationship, so this
    # likely drops nothing in practice, but guards against a row we couldn't build an edge from
    clean_df=df.dropna(subset=['from_node_id', 'to_node_id', 'relationship']).copy()
    
    # Neo4j's driver rejects NaN outright - Task 1 found real nulls in order_id.
    # lets turn every pandas NaN into a plain Python None first
    clean_df=clean_df.where(pd.notnull(clean_df), None)
    
    with driver.session() as session:
        # a constraint enforces uniqueness and makes MERGE fast (it's indexed).
        # every node uses one generic :Node label, with "type" (Customer/Product) stored
        # as a property instead of a second label - Cypher can't parameterize a label name
        session.run("""
            CREATE CONSTRAINT node_id_unique IF NOT EXISTS
            FOR (n:Node) REQUIRE n.id IS UNIQUE
        """)
        
        rows = clean_df.to_dict('records')

        # lets process in batches instead of one giant query.
        for i in range(0, len(rows), BATCH_SIZE):
            batch = rows[i:i + BATCH_SIZE]

            # lets group this batch by relationship type, same reason as the label above,
            # Cypher can't parameterize a relationship TYPE, only property values
            by_relationship = {}
            for row in batch:
                by_relationship.setdefault(row['relationship'], []).append(row)

            for rel_type, rel_rows in by_relationship.items():
                session.run(f"""
                    UNWIND $rows AS row
                    MERGE (a:Node {{id: row.from_node_id}})
                    ON CREATE SET a.type = row.from_node_type
                    MERGE (b:Node {{id: row.to_node_id}})
                    ON CREATE SET b.type = row.to_node_type
                    MERGE (a)-[r:{rel_type}]->(b)
                    SET r.edge_id = row.edge_id,
                        r.timestamp = row.timestamp,
                        r.order_id = row.order_id,
                        r.category = row.category,
                        r.customer_segment = row.customer_segment,
                        r.edge_strength = row.edge_strength,
                        r.price_bucket = row.price_bucket,
                        r.region = row.region,
                        r.state = row.state,
                        r.campaign = row.campaign,
                        r.same_household = row.same_household,
                        r.prior_interactions = row.prior_interactions,
                        r.dwell_seconds = row.dwell_seconds,
                        r.product_id = row.product_id,
                        r.unit_price_usd = row.unit_price_usd,
                        r.quantity = row.quantity,
                        r.returned_flag = row.returned_flag,
                        r.auth_approved = row.auth_approved
                """, rows=rel_rows)

            print(f"   Loaded batch {i // BATCH_SIZE + 1} ({len(batch)} edges)")

    driver.close()
    print(f"   Loaded {len(clean_df)} edges into Neo4j")

def extract_from_neo4j() -> pd.DataFrame:
    """Extract relationships from Neo4j as a tabular edge list."""
    # TODO: Implement using Cypher MATCH query
    driver = neo4j_driver()

    with driver.session() as session:
        # lets walk every (node)-[relationship]->(node) pattern and flatten it into one row per
        # edge - type(r) reads the relationship's actual type back out as a plain string
        result = session.run("""
            MATCH (a:Node)-[r]->(b:Node)
            RETURN
                r.edge_id AS edge_id,
                a.id AS from_node_id,
                a.type AS from_node_type,
                b.id AS to_node_id,
                b.type AS to_node_type,
                type(r) AS relationship,
                r.timestamp AS timestamp,
                r.order_id AS order_id,
                r.category AS category,
                r.customer_segment AS customer_segment,
                r.edge_strength AS edge_strength,
                r.price_bucket AS price_bucket,
                r.region AS region,
                r.state AS state,
                r.campaign AS campaign,
                r.same_household AS same_household,
                r.prior_interactions AS prior_interactions,
                r.dwell_seconds AS dwell_seconds,
                r.product_id AS product_id,
                r.unit_price_usd AS unit_price_usd,
                r.quantity AS quantity,
                r.returned_flag AS returned_flag,
                r.auth_approved AS auth_approved
        """)
        # record.data() turns each result row into a plain dict, then pandas builds
        # the DataFrame from that list of dicts - same end result as the Cassandra version
        df = pd.DataFrame([record.data() for record in result])

    driver.close()
    return df

print("Neo4j functions defined")

Neo4j functions defined


## Task 3.2: Load Source Systems

Load the CSV data into the operational databases (simulating production environment).

In [18]:
# ========= Imports
import os, io, re, time, json, textwrap
from datetime import datetime
from typing import Dict, Any, List, Tuple
import numpy as np
import pandas as pd

# Source system libraries
import psycopg2
from psycopg2.extras import execute_values
from sqlalchemy import create_engine
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from neo4j import GraphDatabase

print("="*60)
print("TASK 3: Loading Source Systems")
print("="*60)

# TODO: Load data into each source system

# Load PostgreSQL
print("\n📦 Loading orders into PostgreSQL...")
pg_load_orders(orders_df)

# Load Cassandra
print("\n📦 Loading events into Cassandra...")
cas_load_events(events_df)

# Load Neo4j
print("\n📦 Loading edges into Neo4j...")
neo4j_load_edges(edges_df)

print("\n✅ All source systems loaded!")

TASK 3: Loading Source Systems

📦 Loading orders into PostgreSQL...
Insertion of records complete

📦 Loading events into Cassandra...
Inserted events in Cassandra

📦 Loading edges into Neo4j...
   Loaded batch 1 (1000 edges)
   Loaded batch 2 (1000 edges)
   Loaded batch 3 (500 edges)
   Loaded 2500 edges into Neo4j

✅ All source systems loaded!


## Task 3.3: Extract and Transform

Extract data from source systems and transform for Redshift staging.

In [19]:
print("\n" + "="*60)
print("Extracting from Source Systems")
print("="*60)

# TODO: Extract data from each source system

print("\n📤 Extracting from PostgreSQL...")
orders_extracted = extract_from_pg()
print(f"   Extracted {len(orders_extracted)} orders")

print("\n📤 Extracting from Cassandra...")
events_extracted = extract_from_cas()
print(f"   Extracted {len(events_extracted)} events")

print("\n📤 Extracting from Neo4j...")
edges_extracted = extract_from_neo4j()
print(f"   Extracted {len(edges_extracted)} edges")

# TODO: Conform columns to staging specs
print("\n🔄 Conforming data to staging specifications...")
orders = orders_extracted[[c for c,_ in ORDERS_COLSPEC if c in orders_extracted.columns]].copy()
events = events_extracted[[c for c,_ in EVENTS_COLSPEC if c in events_extracted.columns]].copy()
edges = edges_extracted[[c for c,_ in EDGES_COLSPEC if c in edges_extracted.columns]].copy()

print("\n✅ Task 3 Complete - Data extracted and transformed!")


Extracting from Source Systems

📤 Extracting from PostgreSQL...
   Extracted 2500 orders

📤 Extracting from Cassandra...
   Extracted 2500 events

📤 Extracting from Neo4j...
   Extracted 2500 edges

🔄 Conforming data to staging specifications...

✅ Task 3 Complete - Data extracted and transformed!


---
# Task 4: Load Data into Redshift

In this task, you will:
- Execute the DDL to create staging, dimension, and fact tables
- Load data into Redshift staging tables
- Populate dimension tables from staging data
- Populate fact tables with dimension key lookups
- Validate successful loading

**Deliverables:**
- Working Redshift connection and execution functions
- Loaded staging, dimension, and fact tables
- Row count validation

## Task 4.1: Define Redshift Functions

In [20]:
# ========= Redshift Functions =========

session_boto = boto3.Session(region_name=AWS_REGION)
rsd = session_boto.client("redshift-data", region_name=AWS_REGION)

def _rs_kwargs() -> Dict[str, Any]:
    """Build Redshift Data API connection parameters."""
    base = dict(Database=REDSHIFT_DATABASE)
    if REDSHIFT_WORKGROUP:
        base["WorkgroupName"] = REDSHIFT_WORKGROUP
        if REDSHIFT_SECRET_ARN:
            base["SecretArn"] = REDSHIFT_SECRET_ARN
    elif REDSHIFT_CLUSTER_IDENTIFIER and REDSHIFT_DB_USER:
        base["ClusterIdentifier"] = REDSHIFT_CLUSTER_IDENTIFIER
        base["DbUser"] = REDSHIFT_DB_USER
    else:
        raise RuntimeError("Configure Redshift serverless OR provisioned for Data API.")
    return base

def rs_exec(sql: str, return_results=False, timeout_s=900):
    """Execute SQL on Redshift via Data API."""
    # TODO: Implement the following steps:
    # 1. Execute statement using rsd.execute_statement()
    # 2. Poll for completion using rsd.describe_statement()
    # 3. If return_results, fetch using rsd.get_statement_result()
    # 4. Return results as list of dicts
    
    # lets submit the sql. **_rskwargs() converts the dict to keyword arguments
    resp=rsd.execute_statement(Sql=sql, **_rs_kwargs())
    
    # every submitted query will get its own ID
    statement_id=resp["Id"]
    
    # poll. Lets record when we started waiting.
    
    start=time.time()
    
    # Lets keep checking until its done. State will be one of: SUBMITTED, PICKED, FINISHED, FAILED, ABORTED
    while True:
        status=rsd.describe_statement(Id=statement_id)
        state=status["Status"]
        
        if state == "FINISHED":
            break
        if state in ("FAILED", "ABORTED"):
            raise RuntimeError(f"Query failed: {status.get('Error', 'unknown error')}")
        
        # safety net - lets not poll forever if something is stuck
        if time.time() - start > timeout_s:
            raise TimeoutError(f"Query timed out after {timeout_s}s")
        
        time.sleep(1)
    
    # some queries like (CREATE, INSERT) dont return rows
    if not return_results:
        return None
    
    # the below fetches the actual rows
    result = rsd.get_statement_result(Id=statement_id)
    
    # ColumnMetadata holds the column names separately from the row data
    columns = [col["name"] for col in result["ColumnMetadata"]]

    rows = []
    
    # every record in the below for loop is a row
    for record in result["Records"]:
        
        row = {}
        # zip() pairs column name #1 with value #1
        for col_name, value in zip(columns, record):
            row[col_name] = None if value.get("isNull") else next(iter(value.values()))
        rows.append(row)
    # return the plain list of dicts
    return rows

def rs_batch_insert(table: str, colspec: List[Tuple[str,str]], df: pd.DataFrame):
    """Load DataFrame into Redshift using batch INSERT statements."""
    # TODO: Implement the following steps:
    # 1. Format values based on column types (s, ts, i, f, b)
    # 2. Build multi-row INSERT statements
    # 3. Execute in batches and report progress
    
    # colspec is a list of tuples.
    col_names = [c for c, _ in colspec]
    
    # the below pulls out the kind letters in the same order. Need to know how to format each collumn's value
    kinds = [k for _, k in colspec]

    
    def format_value(val, kind):
        
        # missing values will become the literal SQL keyword NULL.
        if pd.isna(val):
            return "NULL"
        
        # SQL uses a single quote to end a string so we have to be careful if the string is like O'Brien
        if kind == 's':
            escaped = str(val).replace("'", "''")
            return f"'{escaped}'"
        
        # timstamp also gets wrapped in quotes
        if kind == 'ts':
            return f"'{val}'"
        
        # SQL bool literals are TRUE/FALSE not True/False
        if kind == 'b':
            return "TRUE" if val else "FALSE"
        return str(val)

    # plain list of lists - same .values.tolist() pattern used in the Task 3 loads
    rows = df[col_names].values.tolist()
    
    # Learned that Redshift's data api caps one query at 200KB
    SQL_BATCH_SIZE = 200

    # lets walk through rows in chunks of BATCH_SIZE instead of one giant INSERT
    for i in range(0, len(rows), SQL_BATCH_SIZE):
           
        batch = rows[i:i + SQL_BATCH_SIZE]
        value_rows = []
        for row in batch:
            
            # format every value in this row
            formatted = [format_value(v, k) for v, k in zip(row, kinds)]
            value_rows.append(f"({', '.join(formatted)})")
        
        # lets build one INSERT statement covering this whole batch at once
        sql = f"INSERT INTO {table} ({', '.join(col_names)}) VALUES {', '.join(value_rows)}"
        
        # and run
        rs_exec(sql)
        print(f"   Inserted rows {i} to {i + len(batch)} into {table}")

    print(f"   Loaded {len(df)} rows into {table}")
    

print("Redshift functions defined")

Redshift functions defined


## Task 4.2: Execute DDL and Create Tables

In [21]:
print("="*60)
print("TASK 4: Loading Data into Redshift")
print("="*60)

print("\n📋 Step 1: Executing DDL to create tables...")

# TODO: Read and execute DDL from the markdown file
# 1. Read DDL_MD_PATH file
# 2. Extract SQL blocks from markdown code fences
# 3. Execute each statement (skip CREATE SCHEMA, rewrite schema references)

# Hint: Use regex to extract SQL blocks:
# blocks = re.findall(r"```sql(.*?)```", md_content, flags=re.DOTALL|re.IGNORECASE)

# Hint: Rewrite schema references for public schema:
# s = re.sub(r'\bstg\.', 'public.stg_', s)
# s = re.sub(r'\bdw\.', 'public.dw_', s)


# read the whole DDL markdown file as one string
with open(DDL_MD_PATH, 'r') as f:
    md_content = f.read()

# the actual SQL lives inside ```sql ... ``` fences - pull out just those blocks,
# ignoring the surrounding markdown prose/headings/tables
blocks = re.findall(r"```sql(.*?)```", md_content, flags=re.DOTALL | re.IGNORECASE)

statements_run = 0
statements_skipped = 0

for block in blocks:
    # a fenced block can hold several ; separated statements - split and run one at a time
    for stmt in block.split(';'):
        stmt = stmt.strip()
        if not stmt:
            continue

        # this workspace's Redshift user can create tables/views in "public",
        # but doesn't have permission to create new schemas - lets skip these statements
        if stmt.upper().startswith('CREATE SCHEMA'):
            statements_skipped += 1
            continue
        
        # Rewrite schema references for public schema
        stmt = re.sub(r'\bstg\.', 'public.stg_', stmt)
        stmt = re.sub(r'\bdw\.', 'public.dw_', stmt)

        rs_exec(stmt)
        statements_run += 1

print(f"   Executed {statements_run} DDL statements ({statements_skipped} CREATE SCHEMA skipped)")
print("   Tables created in Redshift as public.stg_* and public.dw_*")



TASK 4: Loading Data into Redshift

📋 Step 1: Executing DDL to create tables...
   Executed 38 DDL statements (2 CREATE SCHEMA skipped)
   Tables created in Redshift as public.stg_* and public.dw_*


## Task 4.3: Load Staging Tables

In [22]:
print("\n📦 Step 2: Loading staging tables...")

# TODO: Load each DataFrame into its staging table

print("\n  Loading orders into stg_orders_raw...")
rs_batch_insert("public.stg_orders_raw", ORDERS_COLSPEC, orders)

print("\n  Loading events into stg_events_raw...")
rs_batch_insert("public.stg_events_raw", EVENTS_COLSPEC, events)

print("\n  Loading edges into stg_edges_raw...")
rs_batch_insert("public.stg_edges_raw", EDGES_COLSPEC, edges)

print("\n✅ Staging tables loaded!")


📦 Step 2: Loading staging tables...

  Loading orders into stg_orders_raw...
   Inserted rows 0 to 200 into public.stg_orders_raw
   Inserted rows 200 to 400 into public.stg_orders_raw
   Inserted rows 400 to 600 into public.stg_orders_raw
   Inserted rows 600 to 800 into public.stg_orders_raw
   Inserted rows 800 to 1000 into public.stg_orders_raw
   Inserted rows 1000 to 1200 into public.stg_orders_raw
   Inserted rows 1200 to 1400 into public.stg_orders_raw
   Inserted rows 1400 to 1600 into public.stg_orders_raw
   Inserted rows 1600 to 1800 into public.stg_orders_raw
   Inserted rows 1800 to 2000 into public.stg_orders_raw
   Inserted rows 2000 to 2200 into public.stg_orders_raw
   Inserted rows 2200 to 2400 into public.stg_orders_raw
   Inserted rows 2400 to 2500 into public.stg_orders_raw
   Loaded 2500 rows into public.stg_orders_raw

  Loading events into stg_events_raw...
   Inserted rows 0 to 200 into public.stg_events_raw
   Inserted rows 200 to 400 into public.stg_events_

## Task 4.4: Populate Dimension Tables

In [23]:
print("\n📊 Step 3: Populating dimension tables...")

# TODO: Populate each dimension table from staging data

# Example for dim_date:
# rs_exec("""
#     INSERT INTO public.dw_dim_date (date_key, date_actual, year, quarter, month, day, week_of_year, day_of_week, is_weekend)
#     SELECT DISTINCT
#         CAST(to_char(dt, 'YYYYMMDD') AS INTEGER) AS date_key,
#         dt AS date_actual,
#         EXTRACT(YEAR FROM dt)::SMALLINT,
#         EXTRACT(QUARTER FROM dt)::SMALLINT,
#         EXTRACT(MONTH FROM dt)::SMALLINT,
#         EXTRACT(DAY FROM dt)::SMALLINT,
#         EXTRACT(WEEK FROM dt)::SMALLINT,
#         EXTRACT(DOW FROM dt)::SMALLINT,
#         (EXTRACT(DOW FROM dt) IN (0,6))::BOOLEAN
#     FROM (
#         SELECT order_datetime::date AS dt FROM public.stg_orders_raw WHERE order_datetime IS NOT NULL
#         UNION SELECT ship_datetime::date FROM public.stg_orders_raw WHERE ship_datetime IS NOT NULL
#         UNION SELECT event_ts::date FROM public.stg_events_raw WHERE event_ts IS NOT NULL
#     ) dates WHERE dt IS NOT NULL;
# """)

# TODO: Populate remaining dimensions:
# - dim_customer (from stg_events_raw and stg_orders_raw)
# - dim_product (from stg_events_raw and stg_edges_raw)
# - dim_channel, dim_device, dim_browser, dim_shipping_method, dim_payment_method, dim_campaign
# - dim_os, dim_referrer, dim_ab_variant

# dim_date
rs_exec("""
    INSERT INTO public.dw_dim_date (date_key, date_actual, year, quarter, month, day, week_of_year, day_of_week, is_weekend)
    SELECT DISTINCT
        CAST(to_char(dt, 'YYYYMMDD') AS INTEGER) AS date_key,
        dt AS date_actual,
        EXTRACT(YEAR FROM dt)::SMALLINT,
        EXTRACT(QUARTER FROM dt)::SMALLINT,
        EXTRACT(MONTH FROM dt)::SMALLINT,
        EXTRACT(DAY FROM dt)::SMALLINT,
        EXTRACT(WEEK FROM dt)::SMALLINT,
        EXTRACT(DOW FROM dt)::SMALLINT,
        (EXTRACT(DOW FROM dt) IN (0,6))::BOOLEAN
    FROM (
        SELECT order_datetime::date AS dt FROM public.stg_orders_raw WHERE order_datetime IS NOT NULL
        UNION SELECT ship_datetime::date FROM public.stg_orders_raw WHERE ship_datetime IS NOT NULL
        UNION SELECT event_ts::date FROM public.stg_events_raw WHERE event_ts IS NOT NULL
    ) dates WHERE dt IS NOT NULL;
""")
print("   dim_date populated")

# dim_customer
rs_exec("""
    INSERT INTO public.dw_dim_customer (
        customer_id, country, state, customer_segment, is_logged_in,
        effective_from, effective_to, is_current
    )
    SELECT
        customer_id,
        MAX(country) AS country,
        MAX(state) AS state,
        NULL AS customer_segment,
        BOOL_OR(is_logged_in) AS is_logged_in,
        GETDATE() AS effective_from,
        NULL AS effective_to,
        TRUE AS is_current
    FROM (
        SELECT customer_id, country, state, NULL::BOOLEAN AS is_logged_in FROM public.stg_orders_raw
        UNION ALL
        SELECT customer_id, country, state, is_logged_in FROM public.stg_events_raw
    ) combined
    GROUP BY customer_id;
""")
print("   dim_customer populated")

# dim_product
rs_exec("""
    INSERT INTO public.dw_dim_product (
        product_id, category, price_bucket, current_unit_price_usd,
        effective_from, effective_to, is_current
    )
    SELECT
        product_id,
        MAX(category) AS category,
        MAX(price_bucket) AS price_bucket,
        MAX(unit_price_usd) AS current_unit_price_usd,
        GETDATE() AS effective_from,
        NULL AS effective_to,
        TRUE AS is_current
    FROM (
        SELECT product_id, category, NULL::VARCHAR(16) AS price_bucket, NULL::DECIMAL(12,2) AS unit_price_usd
        FROM public.stg_events_raw WHERE product_id IS NOT NULL
        UNION ALL
        SELECT product_id, category, price_bucket, unit_price_usd
        FROM public.stg_edges_raw WHERE product_id IS NOT NULL
    ) combined
    GROUP BY product_id;
""")
print("   dim_product populated")


# dim_channel
rs_exec("INSERT INTO public.dw_dim_channel (channel) SELECT DISTINCT channel FROM public.stg_orders_raw WHERE channel IS NOT NULL;")
print("   dim_channel populated")


# dim_device
rs_exec("""
    INSERT INTO public.dw_dim_device (device_type)
    SELECT DISTINCT device_type FROM (
        SELECT device_type FROM public.stg_orders_raw
        UNION SELECT device_type FROM public.stg_events_raw
    ) c WHERE device_type IS NOT NULL;
""")
print("   dim_device populated")

# dim_browser
rs_exec("""
    INSERT INTO public.dw_dim_browser (browser)
    SELECT DISTINCT browser FROM (
        SELECT browser FROM public.stg_orders_raw
        UNION SELECT browser FROM public.stg_events_raw
    ) c WHERE browser IS NOT NULL;
""")
print("   dim_browser populated")

# dim_shipping_method
rs_exec("INSERT INTO public.dw_dim_shipping_method (shipping_method) SELECT DISTINCT shipping_method FROM public.stg_orders_raw WHERE shipping_method IS NOT NULL;")
print("   dim_shipping_method populated")

# dim_payment_method
rs_exec("INSERT INTO public.dw_dim_payment_method (payment_method) SELECT DISTINCT payment_method FROM public.stg_orders_raw WHERE payment_method IS NOT NULL;")
print("   dim_payment_method populated")


# dim_campaign
rs_exec("""
    INSERT INTO public.dw_dim_campaign (campaign)
    SELECT DISTINCT campaign FROM (
        SELECT campaign FROM public.stg_orders_raw
        UNION SELECT campaign FROM public.stg_edges_raw
    ) c WHERE campaign IS NOT NULL;
""")
print("   dim_campaign populated")


# dim_os
rs_exec("INSERT INTO public.dw_dim_os (os) SELECT DISTINCT os FROM public.stg_events_raw WHERE os IS NOT NULL;")
print("   dim_os populated")

# dim_referrer
rs_exec("INSERT INTO public.dw_dim_referrer (referrer) SELECT DISTINCT referrer FROM public.stg_events_raw WHERE referrer IS NOT NULL;")
print("   dim_referrer populated")

# dim_ab_variant
rs_exec("INSERT INTO public.dw_dim_ab_variant (ab_variant) SELECT DISTINCT ab_variant FROM public.stg_events_raw WHERE ab_variant IS NOT NULL;")
print("   dim_ab_variant populated")


print("\n✅ All dimension tables populated!")


📊 Step 3: Populating dimension tables...
   dim_date populated
   dim_customer populated
   dim_product populated
   dim_channel populated
   dim_device populated
   dim_browser populated
   dim_shipping_method populated
   dim_payment_method populated
   dim_campaign populated
   dim_os populated
   dim_referrer populated
   dim_ab_variant populated

✅ All dimension tables populated!


## Task 4.5: Populate Fact Tables

In [24]:
print("\n📊 Step 4: Populating fact tables...")

# TODO: Populate fact tables by joining staging data with dimension surrogate keys

# Example structure for fact_orders:
# rs_exec("""
#     INSERT INTO public.dw_fact_orders (
#         order_id, customer_sk, order_date_key, ship_date_key, channel_sk, ...
#     )
#     SELECT
#         o.order_id,
#         dc.customer_sk,
#         CAST(to_char(o.order_datetime::date, 'YYYYMMDD') AS INTEGER),
#         ...
#     FROM public.stg_orders_raw o
#     LEFT JOIN public.dw_dim_customer dc ON dc.customer_id = o.customer_id AND dc.is_current = TRUE
#     LEFT JOIN public.dw_dim_channel ch ON ch.channel = o.channel
#     ...
# """)

# TODO: Populate:
# - dw_fact_orders
# - dw_fact_events
# - dw_fact_graph_edges

# dw_fact_orders
rs_exec("""
    INSERT INTO public.dw_fact_orders (
        order_id, customer_sk, order_date_key, ship_date_key, channel_sk, device_sk, browser_sk,
        campaign_sk, payment_method_sk, shipping_method_sk, primary_category, num_distinct_items,
        subtotal_usd, discount_rate, discount_amount_usd, shipping_cost_usd, tax_rate, tax_amount_usd,
        order_total_usd, order_weight_kg, delivery_days, on_time_delivery, authorization_approved, returned
    )
    SELECT
        o.order_id,
        dc.customer_sk,
        CAST(to_char(o.order_datetime::date, 'YYYYMMDD') AS INTEGER),
        CAST(to_char(o.ship_datetime::date, 'YYYYMMDD') AS INTEGER),
        ch.channel_sk,
        dv.device_sk,
        br.browser_sk,
        ca.campaign_sk,
        pm.payment_method_sk,
        sm.shipping_method_sk,
        o.primary_category,
        o.num_distinct_items,
        o.subtotal_usd,
        o.discount_rate,
        o.discount_amount_usd,
        o.shipping_cost_usd,
        o.tax_rate,
        o.tax_amount_usd,
        o.order_total_usd,
        o.order_weight_kg,
        o.delivery_days,
        o.on_time_delivery,
        o.authorization_approved,
        o.returned
    FROM public.stg_orders_raw o
    LEFT JOIN public.dw_dim_customer dc ON dc.customer_id = o.customer_id AND dc.is_current = TRUE
    LEFT JOIN public.dw_dim_channel ch ON ch.channel = o.channel
    LEFT JOIN public.dw_dim_device dv ON dv.device_type = o.device_type
    LEFT JOIN public.dw_dim_browser br ON br.browser = o.browser
    LEFT JOIN public.dw_dim_campaign ca ON ca.campaign = o.campaign
    LEFT JOIN public.dw_dim_payment_method pm ON pm.payment_method = o.payment_method
    LEFT JOIN public.dw_dim_shipping_method sm ON sm.shipping_method = o.shipping_method;
""")
print("   dw_fact_orders populated")

# dw_fact_events
rs_exec("""
    INSERT INTO public.dw_fact_events (
        event_id, customer_sk, product_sk, event_date_key, session_id, event_type,
        channel_sk, device_sk, browser_sk, os_sk, referrer_sk, ab_variant_sk,
        page_depth, latency_ms, dwell_seconds, cart_value_usd, discount_rate, fraud_score,
        payment_outcome, sequence_num, category, promo_code
    )
    SELECT
        e.event_id,
        dc.customer_sk,
        dp.product_sk,
        CAST(to_char(e.event_ts::date, 'YYYYMMDD') AS INTEGER),
        e.session_id,
        e.event_type,
        NULL,
        dv.device_sk,
        br.browser_sk,
        dos.os_sk,
        rf.referrer_sk,
        ab.ab_variant_sk,
        e.page_depth,
        e.latency_ms,
        e.dwell_seconds,
        e.cart_value_usd,
        e.discount_rate,
        e.fraud_score,
        e.payment_outcome,
        e.sequence_num,
        e.category,
        e.promo_code
    FROM public.stg_events_raw e
    LEFT JOIN public.dw_dim_customer dc ON dc.customer_id = e.customer_id AND dc.is_current = TRUE
    LEFT JOIN public.dw_dim_product dp ON dp.product_id = e.product_id AND dp.is_current = TRUE
    LEFT JOIN public.dw_dim_device dv ON dv.device_type = e.device_type
    LEFT JOIN public.dw_dim_browser br ON br.browser = e.browser
    LEFT JOIN public.dw_dim_os dos ON dos.os = e.os
    LEFT JOIN public.dw_dim_referrer rf ON rf.referrer = e.referrer
    LEFT JOIN public.dw_dim_ab_variant ab ON ab.ab_variant = e.ab_variant;
""")
print("   dw_fact_events populated")

# dw_fact_graph_edges
rs_exec("""
    INSERT INTO public.dw_fact_graph_edges (
        edge_id, event_date_key, relationship, from_customer_sk, to_customer_sk,
        from_product_sk, to_product_sk, order_id, category, campaign_sk, customer_segment,
        region, state, edge_strength, price_bucket, prior_interactions, dwell_seconds,
        unit_price_usd, quantity, returned_flag, auth_approved
    )
    SELECT
        ed.edge_id,
        CAST(to_char(ed."timestamp"::date, 'YYYYMMDD') AS INTEGER),
        ed.relationship,
        fc.customer_sk,
        tc.customer_sk,
        fp.product_sk,
        tp.product_sk,
        ed.order_id,
        ed.category,
        ca.campaign_sk,
        ed.customer_segment,
        ed.region,
        ed.state,
        ed.edge_strength,
        ed.price_bucket,
        ed.prior_interactions,
        ed.dwell_seconds,
        ed.unit_price_usd,
        ed.quantity,
        ed.returned_flag,
        ed.auth_approved
    FROM public.stg_edges_raw ed
    LEFT JOIN public.dw_dim_customer fc ON fc.customer_id = ed.from_node_id AND ed.from_node_type = 'Customer' AND fc.is_current = TRUE
    LEFT JOIN public.dw_dim_customer tc ON tc.customer_id = ed.to_node_id AND ed.to_node_type = 'Customer' AND tc.is_current = TRUE
    LEFT JOIN public.dw_dim_product fp ON fp.product_id = ed.from_node_id AND ed.from_node_type = 'Product' AND fp.is_current = TRUE
    LEFT JOIN public.dw_dim_product tp ON tp.product_id = ed.to_node_id AND ed.to_node_type = 'Product' AND tp.is_current = TRUE
    LEFT JOIN public.dw_dim_campaign ca ON ca.campaign = ed.campaign;
""")
print("   dw_fact_graph_edges populated")

print("\n✅ Task 4 Complete - All data loaded into Redshift!")


📊 Step 4: Populating fact tables...
   dw_fact_orders populated
   dw_fact_events populated
   dw_fact_graph_edges populated

✅ Task 4 Complete - All data loaded into Redshift!


---
# Task 5: Optimize Performance and Build OLAP Structures

In this task, you will:
- Verify distribution styles and sort keys are applied
- Run ANALYZE to update statistics
- Create materialized views for common queries

**Deliverables:**
- At least one materialized view for common analytics
- ANALYZE run on key tables

In [25]:
print("="*60)
print("TASK 5: Optimize Performance")
print("="*60)

# TODO: Create materialized view for daily revenue
print("\n📊 Creating materialized view for daily revenue...")

# Note: Redshift doesn't support "IF NOT EXISTS" for materialized views
# Use DROP + CREATE pattern:
rs_exec("DROP MATERIALIZED VIEW IF EXISTS public.dw_mv_daily_revenue;")
rs_exec("""
    CREATE MATERIALIZED VIEW public.dw_mv_daily_revenue AS
    SELECT
        order_date_key,
        COUNT(*) AS orders,
        SUM(order_total_usd) AS revenue_usd,
        AVG(order_total_usd) AS avg_order_value
    FROM public.dw_fact_orders
    GROUP BY order_date_key;
""")


TASK 5: Optimize Performance

📊 Creating materialized view for daily revenue...


In [26]:
# TODO: Run ANALYZE on key tables
print("\n📊 Running ANALYZE on tables...")

for table in ['dw_fact_orders', 'dw_fact_events', 'dw_fact_graph_edges', 
              'dw_dim_customer', 'dw_dim_product', 'dw_dim_date']:
    rs_exec(f"ANALYZE public.{table};")
    print(f"  ✓ ANALYZE complete: {table}")

print("\n✅ Task 5 Complete - Performance optimization done!")


📊 Running ANALYZE on tables...
  ✓ ANALYZE complete: dw_fact_orders
  ✓ ANALYZE complete: dw_fact_events
  ✓ ANALYZE complete: dw_fact_graph_edges
  ✓ ANALYZE complete: dw_dim_customer
  ✓ ANALYZE complete: dw_dim_product
  ✓ ANALYZE complete: dw_dim_date

✅ Task 5 Complete - Performance optimization done!


---
# Task 6: Validate and Report Your Results

In this task, you will:
- Run data quality checks
- Execute sample analytical queries
- Generate the final report

**Deliverables:**
- Data quality checks (row counts, null checks)
- Sample analytical query results
- Final report with schema diagram and design rationale

In [27]:
print("="*60)
print("TASK 6: Validation and Reporting")
print("="*60)

# TODO: Query row counts for all tables
print("\n📊 Row Counts:")
print("-" * 40)

# Example:
# row_counts = rs_exec("""
#     SELECT 'stg_orders_raw' AS table_name, COUNT(*) AS n FROM public.stg_orders_raw
#     UNION ALL SELECT 'stg_events_raw', COUNT(*) FROM public.stg_events_raw
#     UNION ALL SELECT 'dw_fact_orders', COUNT(*) FROM public.dw_fact_orders
#     ...
# """, return_results=True)
# 

row_counts = rs_exec("""
    SELECT 'stg_orders_raw' AS table_name, COUNT(*) AS n FROM public.stg_orders_raw
    UNION ALL SELECT 'stg_events_raw', COUNT(*) FROM public.stg_events_raw
    UNION ALL SELECT 'stg_edges_raw', COUNT(*) FROM public.stg_edges_raw
    UNION ALL SELECT 'dw_dim_date', COUNT(*) FROM public.dw_dim_date
    UNION ALL SELECT 'dw_dim_customer', COUNT(*) FROM public.dw_dim_customer
    UNION ALL SELECT 'dw_dim_product', COUNT(*) FROM public.dw_dim_product
    UNION ALL SELECT 'dw_dim_campaign', COUNT(*) FROM public.dw_dim_campaign
    UNION ALL SELECT 'dw_dim_channel', COUNT(*) FROM public.dw_dim_channel
    UNION ALL SELECT 'dw_dim_device', COUNT(*) FROM public.dw_dim_device
    UNION ALL SELECT 'dw_dim_browser', COUNT(*) FROM public.dw_dim_browser
    UNION ALL SELECT 'dw_dim_os', COUNT(*) FROM public.dw_dim_os
    UNION ALL SELECT 'dw_dim_referrer', COUNT(*) FROM public.dw_dim_referrer
    UNION ALL SELECT 'dw_dim_shipping_method', COUNT(*) FROM public.dw_dim_shipping_method
    UNION ALL SELECT 'dw_dim_payment_method', COUNT(*) FROM public.dw_dim_payment_method
    UNION ALL SELECT 'dw_dim_ab_variant', COUNT(*) FROM public.dw_dim_ab_variant
    UNION ALL SELECT 'dw_fact_orders', COUNT(*) FROM public.dw_fact_orders
    UNION ALL SELECT 'dw_fact_events', COUNT(*) FROM public.dw_fact_events
    UNION ALL SELECT 'dw_fact_graph_edges', COUNT(*) FROM public.dw_fact_graph_edges
""", return_results=True)

for r in row_counts:
    print(f"  {r['table_name']}: {r['n']:,}")


TASK 6: Validation and Reporting

📊 Row Counts:
----------------------------------------
  dw_dim_referrer: 6
  stg_orders_raw: 2,500
  dw_fact_orders: 2,500
  dw_dim_os: 5
  dw_dim_shipping_method: 3
  dw_dim_device: 3
  dw_dim_browser: 5
  dw_fact_events: 2,500
  stg_events_raw: 2,500
  dw_dim_ab_variant: 2
  dw_dim_customer: 4,849
  dw_dim_channel: 5
  dw_fact_graph_edges: 2,500
  stg_edges_raw: 2,500
  dw_dim_date: 552
  dw_dim_campaign: 7
  dw_dim_product: 3,107
  dw_dim_payment_method: 5


In [28]:
# TODO: Run sample analytical queries
print("\n📊 Sample Analytics - Daily Revenue:")
print("-" * 40)

# Example query using materialized view:
daily_rev = rs_exec("""
    SELECT d.date_actual, mv.revenue_usd, mv.orders, mv.avg_order_value
    FROM public.dw_mv_daily_revenue mv
    JOIN public.dw_dim_date d ON d.date_key = mv.order_date_key
    ORDER BY d.date_actual
    LIMIT 10;
""", return_results=True)

display(pd.DataFrame(daily_rev))



📊 Sample Analytics - Daily Revenue:
----------------------------------------


,date_actual,revenue_usd,orders,avg_order_value
0,2024-01-01,4086.97,6,681.16
1,2024-01-02,2150.01,5,430.00
2,2024-01-03,681.74,6,113.62
3,2024-01-04,731.67,3,243.89
4,2024-01-05,1302.17,4,325.54
5,2024-01-06,4421.65,9,491.29
6,2024-01-07,2293.13,5,458.62
7,2024-01-09,1712.60,6,285.43
8,2024-01-10,2472.00,8,309.00
9,2024-01-11,4127.43,6,687.90


In [29]:
# TODO: Generate final report
print("\n📄 Generating Final Report...")

# Create a markdown report with:
# - Schema diagram (reference project-mermaid-diagram.md)
# - Design rationale
# - Row counts for all tables
# - Sample query results

# lets split the row counts into three: staging_counts, fact_counts, dim_counts
staging_counts = [r for r in row_counts if r['table_name'].startswith('stg_')]
fact_counts = [r for r in row_counts if r['table_name'].startswith('dw_fact_')]
dim_counts = [r for r in row_counts if r['table_name'].startswith('dw_dim_')]

# lets convert the list of dicts into plain text lines, like "stg_orders_raw: 2500 rows"
staging_lines = "\n".join(f"- `{r['table_name']}`: {r['n']:,} rows" for r in staging_counts)
fact_lines = "\n".join(f"- `{r['table_name']}`: {r['n']:,} rows" for r in fact_counts)
dim_lines = "\n".join(f"- `{r['table_name']}`: {r['n']:,} rows" for r in dim_counts)

# lets build the whole report as one string - {staging_lines}/{fact_lines}/{dim_lines}
report_content = f"""# Data Warehouse Build Report

Generated: {datetime.utcnow().isoformat()}Z

## Schema Overview

### Staging Tables

{staging_lines}

### Fact Tables

{fact_lines}

### Dimension Tables

{dim_lines}

## Design Rationale


### 1. Why star schema?

Three unrelated source systems each describe a different kind of occurence:

- orders (PostgreSQL) - one row is one order, like customer C55 buying 3 items on a ceratin date for certain amount
- events (Cassandra) - one row is one click or action, like customer C55 viewing a product page
- edges (Neo4J) - one row is one relationship, like customer C55 having PURCHASED product

These are genuinely differnt shapes of data, so a star scehma keeps each one in its own fact table (fact_orders,
fact_events, fact_graph_edges) instead of forcing them into a single table with mostly empty columns depdending 
on which kind of row it is.

Shared context lives once in a dimension table and gets refrenced from every fact table that needs it:

dim_customer holds one row for customer c55, and all three fact tables point back to that same row through customer_sk
instead of each fact table keeping its own copy.

dim_date holds one row per calendar day, referenceed by order date, ship date, event date, edge date across all 
three fact tables intead of three separate calendars



### 2. Distribution key choices


Customer-centric facts use DISTKEY(customer_sk)

- fact_orders and fact_events both use DISTKEY(customer_sk)
- Example: customer C55 (customer_sk = 205) - every order row and every event rrow for C55 lands on the same physical
node, since they all share that same customer_sk value
- Result: fact_orders and fact_events colocate with each other, and a query like "total spend per customer" grop by
customer_sk is fast, since one customer's rows area already sitting together


Product-centric fact uses DISTKEY(to_product_sk)

- fact_graph_edges uses DISTKEY(to_product_sk) instead, since its main analytical angle is product relationships, not customers
- Same applies — dim_product's own DISTKEY is product_id, not product_sk

The 9 small dimensions use DISTSTYLE ALL

- Example: dim_channel has only 5 rows — copying all 5 rows to every single node costs almost nothing
- That means joining any fact table to dim_channel never needs a shuffle, since the whole table is already sitting locally on every node


### 3. Sort key choices

Every fact table is sorted by its date key

- fact_orders - sorted by order_date_key
- fact_events - sorted by event_date_key
- fact_graph_edges - sorted by event_date_key

Why date specifically, and not some other column — this warehouse is built for time-based analytics 
(daily revenue, monthly trends, this quarter vs last quarter), so the date key is the column almost every real 
query is going to filter or group by. Sorting by the column your queries actually use is the whole point
— sorting by, say, channel instead wouldn't help a "revenue this month" query at all.

### 4. Materialized view purpose

- dw_mv_daily_revenue takes fact_orders (2500 rows) and pre-computes, per day: how many orders, total revenue,
average order value
- Result: instead of 2500 individual order rows, we get one row per distinct day (roughly 552 rows, matching dim_date's count)
 each row already holding the answer

Regular view vs. materialized view:

- A regular VIEW is just a saved query — every time you SELECT from it, Redshift re-runs the whole aggregation 
from scratch, scanning all 2500 rows again
- A materialized view actually stores the computed result physically. Reading from it means reading pre-computed 
numbers, not recalculating them

Example:

- Someone asks "what was revenue on June 12, 2024?"
- Without the materialized view: Redshift scans all 2500 fact_orders rows, filters to June 12, 
sums them up — every single time this question gets asked
- With the materialized view: Redshift just reads the one row for June 12 that's already sitting there with 
the answer computed — no scanning, no summing


## Analytics Capabilities


- Revenue/order-volume by day/week/month/quarter — via dim_date

- Customer-level analysis — via dim_customer, joined across fact_orders and fact_events

- Product-level analysis — via dim_product, joined across fact_events and fact_graph_edges

- Channel/device/browser/campaign breakdowns — for marketing/UX

- Graph-relationship analysis — via fact_graph_edges

- Pre-aggregated daily revenue — via dw_mv_daily_revenue


"""





# Save report
report_path = os.path.join(BASE_DIR, "warehouse_report.md")
with open(report_path, "w") as f:
    f.write(report_content)

print(f"\n✅ Report saved to: {report_path}")
print("\n" + "="*60)
print("ALL TASKS COMPLETED! ✅")
print("="*60)


📄 Generating Final Report...

✅ Report saved to: ./warehouse_report.md

ALL TASKS COMPLETED! ✅
